# Feature extraction: HuBERT Base, and the re-extraction check of WavLM Base

Run after `notebooks/02_migrate_wavlm_features.ipynb` has passed, at a fixed
commit, on a Colab runtime with a T4 GPU.

This notebook extracts the unpadded features of HuBERT Base
(`facebook/hubert-base-ls960` at the revision of the registry, eager
attention), one condition per run, with the functions and the shard format
of the WavLM Base features. Two values are set in the cell "What this run
does":

- `CONDITION = ""`: the short check only. HuBERT Base is loaded and compared
  with its registry entry, the file lists are compared with the files of the
  WavLM Base shards, and 25 files of every split are encoded in every
  condition, checked and timed. The expected time and size of the extraction
  of each condition are shown. No feature shard is written.
- `CONDITION = "clean"` (or `"trim40"`, `"trim20"`, `"trim15"`): the short
  check for that condition, then the extraction of that condition into
  `features/hubert-base/unpadded/<condition>/` of the new root, then the
  check of the written shards.
- `WAVLM_CHECK = True` repeats the re-extraction check of WavLM Base first
  (200 files of every split in every condition against the stored features).
  It has passed on Colab and is not needed for the extraction.

The extraction is resumable: a shard (2,000 files) is written when it is
complete, and an existing shard is kept. After a disconnect the notebook is
run again from the top with the same `CONDITION`; it continues with the
first shard that does not exist. Every shard records the encoder, the
checkpoint revision, the library versions, the commit of this repository
and the GPU.

No shard of WavLM Base is written, changed or deleted; the WavLM Base
features are only read. Check tables are written under `checks/`.

1. the datasets are unpacked to the local disk from the archives on the new root;
2. (only with `WAVLM_CHECK`) the re-extraction check of WavLM Base, with its summary and hard stop;
3. HuBERT Base is loaded at the pinned revision and checked against its registry entry;
4. short check: file lists, frame counts, lengths and flags, timing (hard stop if any row fails);
5. (only with a `CONDITION`) the extraction of that condition;
6. (only with a `CONDITION`) the written shards against the WavLM Base shards and their own meta (hard stop).

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

In [ ]:
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

### What this run does

Leave `CONDITION` empty for the short check. Set it to one condition to
extract that condition; one condition per run.

In [ ]:
CONDITION = ""         # "": the short check only; "clean", "trim40", "trim20" or "trim15": extract this condition
WAVLM_CHECK = False    # True: repeat the re-extraction check of WavLM Base first

ENCODER = "hubert-base"       # the encoder that is extracted
REFERENCE = "wavlm-base"      # its stored features give the files, their order, their lengths and their flags
assert CONDITION == "" or CONDITION in cfg.conditions(), \
    f"CONDITION is {CONDITION!r}: leave it empty or set it to one of {list(cfg.conditions())}"
print("this run:", f"extraction of {ENCODER}, condition {CONDITION}" if CONDITION else f"short check of {ENCODER}",
      "| re-extraction check of", REFERENCE, "first:", WAVLM_CHECK)

### Datasets

ASVspoof 2019 LA and In-the-Wild are unpacked to the local disk of the
runtime from the archives on the new root (In-the-Wild only if the archive
has the sha256 of `config.ITW_ZIP_SHA256`). The file lists are checked; their
order is the order of the files in the shards.

In [ ]:
from ssl_silence import datasets, encoders, extraction, shards

datasets.ensure_asvspoof(cfg.la_zip())
datasets.ensure_itw(cfg.itw_zip(), sha256=cfg.ITW_ZIP_SHA256)
file_lists = {}
for split in cfg.SPLITS:
    file_lists[split] = datasets.items(split)
    print(split, datasets.check_items(file_lists[split], split))

### Re-extraction check of WavLM Base (only with `WAVLM_CHECK`)

The WavLM Base checkpoint is loaded at the revision of the registry and
compared with its entry (`encoders.load_encoder`). The same positions of a
file list are drawn in every condition (`config.REEXTRACTION_FILES` files
from the generator stream `config.REEXTRACTION_SEED`). Each file goes
through the functions of the full extraction, one file at a time.

One row per split and key. A feature tensor passes if its largest absolute
difference from the stored tensor is at most `verify.FEATURE_TOLERANCE`
(1e-4); `exactly_zero` says whether it is 0. The lengths, frame counts,
flags, labels and ids must be the stored values. The padded shards hold one
layer, and that layer is compared.

All tables are written before the notebook stops, so that every failing
combination is visible. The shards are read from Drive, which takes most of
the time.

In [ ]:
import torch

assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again from the top"
DEVICE = "cuda"
if WAVLM_CHECK:
    reference_entry = cfg.encoder(REFERENCE)
    reference_model = encoders.load_encoder(REFERENCE, DEVICE)
    described = {key: str(value) for key, value in encoders.describe(reference_model).items()}
    record("reextraction_encoder", pd.DataFrame([{"encoder": REFERENCE, "hf_id": reference_entry["hf_id"],
                                                  "revision": reference_entry["revision"],
                                                  "gpu": provenance.gpu_name(DEVICE), **described}]), stop=False)

In [ ]:
if WAVLM_CHECK:
    padded_conditions = list(legacy["features"][REFERENCE]["padded"]["folders"])
    COMBINATIONS = [("unpadded", condition) for condition in cfg.conditions()]
    COMBINATIONS += [("padded", condition) for condition in padded_conditions]
    print("files per split and condition:", cfg.REEXTRACTION_FILES, "| tolerance:", verify.FEATURE_TOLERANCE)

    for kind, condition in COMBINATIONS:
        folder = cfg.feature_dir(REFERENCE, kind, condition)
        parts = []
        for split in cfg.SPLITS:
            parts.append(extraction.reextraction_check(file_lists[split], folder, split, model=reference_model,
                                                       extraction=kind, condition=condition, device=DEVICE))
            print(kind, condition, split, "done")
        record(f"reextraction_{kind}_{condition}", pd.concat(parts, ignore_index=True), stop=False)

In [ ]:
if WAVLM_CHECK:
    tables = {name: table for name, table in checks.items() if name.startswith("reextraction_") and "ok" in table.columns}
    summary = verify.summarise_checks(tables)
    display(summary)
    print("check tables written to", CHECKS)
    for name, table in tables.items():
        verify.require_ok(table, name)
    assert len(tables) == len(COMBINATIONS) and summary["ok"].all()
    print("re-extraction check passed:", REFERENCE, "at revision", reference_entry["revision"])
    del reference_model
    torch.cuda.empty_cache()

### HuBERT Base

The checkpoint is downloaded from Hugging Face at the revision of the
registry (`src/ssl_silence/configs/encoders.yaml`) and compared with its
entry: model class, attention implementation, number of layers, hidden
size, normalisation of the convolutional encoder, kernel and stride lengths,
float32, evaluation mode, no trainable parameter (`encoders.load_encoder`
stops otherwise). The analysis plan asserts 12 layers and group
normalisation.

In [ ]:
entry = cfg.encoder(ENCODER)
model = encoders.load_encoder(ENCODER, DEVICE)
described = encoders.describe(model)
print("encoder:", ENCODER, "|", entry["hf_id"], "| revision:", entry["revision"])
print("feat_extract_norm:", described["feat_extract_norm"], "| transformer layers:", described["n_layers"],
      "| hidden size:", described["hidden_size"], "| attention:", described["attn_implementation"])
assert described["feat_extract_norm"] == "group" and described["n_layers"] == 12, described
RUN_NAME = CONDITION or "all"
record(f"extraction_{ENCODER}_encoder_{RUN_NAME}", pd.DataFrame([{
    "encoder": ENCODER, "hf_id": entry["hf_id"], "revision": entry["revision"], "git_commit": HEAD,
    "gpu": provenance.gpu_name(DEVICE), **{key: str(value) for key, value in described.items()},
    **provenance.versions()}]), stop=False)

### Short check

For the condition of this run, or for all four conditions when `CONDITION`
is empty:

- file lists: every split has the files of the WavLM Base shards, in the
  same order, with the same labels (all files; the ids, labels, lengths and
  flags of the WavLM Base shards are read from Drive, which takes most of
  the time of this cell);
- `config.SHORT_CHECK_FILES` files of every split, drawn from the generator
  stream of the re-extraction check, go through the functions of the full
  extraction, one at a time. For every file: the number of frames equals the
  output length of the convolutional encoder for the encoder input
  (`encoders.conv_output_length`); the pooled features have the shape
  (layers, hidden size) and are finite; the number of real samples, the
  fallback flag and the too-short flag are those stored for WavLM Base;
- estimate: the mean time per file of a split times its number of files,
  summed over the splits, and the size of the two feature tensors of all
  files. `reference_folder_gb` is the size on Drive of the WavLM Base folder
  of the same condition. GB are 10^9 bytes.

All tables are written before the cell stops.

In [ ]:
CHECKED = [CONDITION] if CONDITION else list(cfg.conditions())
sizes = {split: len(file_lists[split]) for split in cfg.SPLITS}
references, lists, parts, folder_gb = {}, [], [], []
for condition in CHECKED:
    reference_folder = cfg.feature_dir(REFERENCE, "unpadded", condition)
    references[condition] = shards.shard_lengths(reference_folder)
    folder_gb.append(sum(os.path.getsize(os.path.join(reference_folder, name))
                         for name in os.listdir(reference_folder)) / 1e9)
    for split in cfg.SPLITS:
        lists.append(extraction.file_list_check(file_lists[split], references[condition], split)
                     .assign(condition=condition))
        parts.append(extraction.short_check(file_lists[split], references[condition], split, model=model,
                                            condition=condition, device=DEVICE))
    print(condition, "checked")
file_list_table = pd.concat(lists, ignore_index=True)
short = pd.concat(parts, ignore_index=True)
record(f"extraction_{ENCODER}_file_lists_{RUN_NAME}", file_list_table, stop=False)
record(f"extraction_{ENCODER}_short_check_{RUN_NAME}", short, stop=False)
estimate = extraction.extraction_estimate(short, sizes, entry["n_layers"], entry["hidden_size"])
estimate["reference_folder_gb"] = folder_gb
record(f"extraction_{ENCODER}_estimate_{RUN_NAME}", estimate, stop=False)

verify.require_ok(file_list_table, "file lists against the shards of " + REFERENCE)
verify.require_ok(short, "short check of " + ENCODER)
assert len(short) == len(CHECKED) * len(cfg.SPLITS) * cfg.SHORT_CHECK_FILES, len(short)
print("short check passed:", ENCODER, "| conditions:", CHECKED, "| files encoded:", len(short),
      "| frame counts equal to the conv output length:", bool((short["n_frames"] == short["conv_frames"]).all()))
for row in estimate.itertuples():
    print(f"expected for {row.condition}: {row.hours:.1f} h on {provenance.gpu_name(DEVICE)}, "
          f"{row.feature_gb:.1f} GB in {row.n_shards} shards "
          f"(the folder of {REFERENCE}: {row.reference_folder_gb:.1f} GB)")
print("all conditions of this run: %.1f h, %.1f GB" % (estimate["hours"].sum(), estimate["feature_gb"].sum()))

### Extraction of one condition (only with a `CONDITION`)

`extraction.extract_shards` with the loaded model, split by split. A shard
that exists is kept; a shard is written to a temporary file and renamed
when it is complete.

In [ ]:
if CONDITION:
    FOLDER = cfg.feature_dir(ENCODER, "unpadded", CONDITION)
    print("free on Google Drive: %.1f GB" % cfg.drive_quota()["free_gb"], "| writing to", FOLDER)
    for split in cfg.SPLITS:
        paths = extraction.extract_shards(file_lists[split], FOLDER, split, encoder=ENCODER,
                                          extraction="unpadded", condition=CONDITION, device=DEVICE, model=model)
        print(split, "| shards:", len(paths), "of", shards.n_shards(sizes[split], cfg.SHARD_SIZE))
else:
    print("CONDITION is empty: nothing is extracted in this run")

### The written shards (only with a `CONDITION`)

- every file of the HuBERT Base shards against the WavLM Base shards of the
  same condition: the same ids in the same order, the same labels, the same
  number of real samples, the same fallback flag and too-short flag (so the
  fallback counts per split and class are equal too);
- what the shards record: one row per distinct combination of encoder,
  revision, attention implementation, commit, GPU and library versions. A
  row passes if it has the registry entry, this condition, a commit without
  uncommitted changes and the library versions of this runtime.

In [ ]:
if CONDITION:
    lengths = extraction.lengths_check(shards.shard_lengths(FOLDER), references[CONDITION])
    record(f"extraction_{ENCODER}_lengths_{CONDITION}", lengths, stop=False)
    shard_meta = extraction.shard_meta_check(extraction.shard_meta_table(FOLDER), ENCODER, CONDITION)
    record(f"extraction_{ENCODER}_shards_{CONDITION}", shard_meta, stop=False)
    print("check tables written to", CHECKS)
    verify.require_ok(lengths, "files, lengths and flags against " + REFERENCE)
    verify.require_ok(shard_meta, "what the shards record")
    assert int(shard_meta["n_files"].sum()) == sum(sizes.values()), "not every file is in a shard"
    written_gb = sum(os.path.getsize(os.path.join(FOLDER, name)) for name in os.listdir(FOLDER)) / 1e9
    print("extraction finished:", ENCODER, "| condition:", CONDITION, "| files:", int(shard_meta["n_files"].sum()),
          "| shards:", int(shard_meta["n_shards"].sum()), "| fallback files:", int(lengths["n_fallback"].sum()),
          "| folder: %.1f GB" % written_gb)